# Silver feature-engineering readiness index

Answer-first campaign report over the complete current Silver registry. It summarizes readiness, blockers, existing feature coverage, and review-only feature opportunities. It contains no target correlations and reads no old Gold or model-ready data.

In [ ]:
_PAYLOAD_B64 = "eNpVT8tOxDAM/Befe+DcX0HIMs4QRbhxlBhKtdp/Jy27B27WvDxzI5UQ80zrjULeDWNe9/tCrfs3qlTFSalsTUquXBKtFNiaSYCmrkNSqRinj6SKHaMMHurtMma3xOo1umgMLlXtK2GGvCxkyKIHX5IzhtYPsYGFNk+wCzr+EyE9I1h26eDnswc7u+CnQQOJn0te32ZB3x/X7v2TvSf0C5iGMfV/g38BUr5g0Q=="
import base64, json, zlib
import pandas as pd
from IPython import get_ipython

_IPYTHON = get_ipython()
if _IPYTHON is not None:
    _IPYTHON.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
P = json.loads(zlib.decompress(base64.b64decode(_PAYLOAD_B64)).decode("utf-8"))
READINESS, CATALOG, SPECS, PROVENANCE = P["readiness"], P["catalog"], P["specs"], P["provenance"]
ROWS = pd.DataFrame(READINESS["rows"])
pd.set_option("display.max_rows", 20); pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 120, "axes.grid": True, "grid.alpha": .2})


## Campaign decision capsule

The campaign decision is complete only when every current Silver table has a validated dossier.

In [ ]:

expected = READINESS["expected_tables"]
actual = sorted(ROWS.table_name.tolist())
complete = actual == sorted(expected) and len(actual) == 42
capsule = pd.DataFrame([{
    "dossiers": len(actual), "expected": len(expected), "complete": complete,
    "ready_for_feature_ideation": int((ROWS.disposition == "ready_for_feature_ideation").sum()),
    "needs_contract_or_data_fix": int((ROWS.disposition == "needs_contract_or_data_fix").sum()),
    "diagnostic_only": int((ROWS.disposition == "diagnostic_only").sum()),
    "blocked": int((ROWS.disposition == "blocked").sum()),
    "excluded_leakage": int((ROWS.disposition == "excluded_leakage").sum()),
    "candidate_count": int(ROWS.candidate_count.sum()),
    "prototype_ready_candidate_count": int(ROWS.get("prototype_ready_candidate_count", pd.Series(dtype=int)).sum()),
    "candidate_prerequisite_count": int(ROWS.get("candidate_prerequisite_count", pd.Series(dtype=int)).sum()),
    "campaign_id": PROVENANCE.get("campaign_id"), "analysis_scope": "source-only Silver",
    "exactness": "exact",
}])
display(capsule)
assert complete, {"missing": sorted(set(expected)-set(actual)), "extra": sorted(set(actual)-set(expected))}
display(Markdown("**Decision.** The readiness index contains exactly 42/42 current Silver dossiers. Candidate promotion remains a separate reviewed action."))


## Registry and dossier coverage

Exact registry-set equality prevents stale, missing, extra, Gold, or cross-campaign artifacts from being summarized.

In [ ]:

display(ROWS.sort_values("table_name"))
display(pd.DataFrame([{"field": k, "value": json.dumps(v, sort_keys=True) if isinstance(v, (dict,list)) else v, "exactness": "exact"} for k,v in sorted(PROVENANCE.items())]))
assert READINESS["analysis_scope"]["gold_contracts_included"] == 0


## Readiness and blockers

Disposition and blocker distributions show where contract/data work is required before prototyping.

In [ ]:

counts = ROWS.disposition.value_counts().sort_index()
ax = counts.plot.bar(color="#356AA0", figsize=(8, 3.5), legend=False)
ax.set_title(f"Silver readiness dispositions — n={len(ROWS)} dossiers — exact")
ax.set_ylabel("Tables"); plt.xticks(rotation=35, ha="right"); plt.show()
display(ROWS.loc[ROWS.blocker_count.gt(0), ["table_name","disposition","analysis_exactness","blocker_count","finding_count","candidate_count"]].sort_values(["blocker_count","table_name"], ascending=[False,True]))


## Existing feature coverage

Existing source keys and table-level feature-family references show current coverage without claiming unverified candidate or column parity.

In [ ]:

coverage = pd.DataFrame([{"table_name": name,
                          "source_keys": spec.get("source_keys", []),
                          "existing_feature_families": spec.get("existing_feature_families", []),
                          "column_coverage_status": spec.get("existing_feature_coverage", {}).get("column_coverage_status", "not_assessed"),
                          "candidate_classification_verified": spec.get("existing_feature_coverage", {}).get("candidate_classification_verified", False),
                          "feature_disposition": spec.get("feature_disposition")}
                         for name, spec in sorted(SPECS.items())])
coverage["existing_family_count"] = coverage.existing_feature_families.map(len)
coverage["exactness"] = "exact"
display(coverage)
ax = coverage.sort_values(["existing_family_count","table_name"]).tail(20).plot.barh(
    x="table_name", y="existing_family_count", color="#356AA0", legend=False, figsize=(9,6))
ax.set_title(
    f"Table-level feature-family references — n={len(coverage)} Silver sources; "
    "top 20 shown — exact"
)
ax.set_xlabel("Mapped feature families"); plt.show()


## Review-only candidate catalog

All candidates are unpromoted review records with evidence, counter-evidence, PIT rules, and independent review status.

In [ ]:

candidate_rows = []
for table, block in sorted(CATALOG["tables"].items()):
    candidate_rows.extend(block.get("candidates", []))
candidates = pd.DataFrame(candidate_rows)
if candidates.empty:
    display(pd.DataFrame([{"status": "No campaign candidates", "exactness": "exact"}]))
else:
    candidates["exactness"] = "exact"
    display(candidates[[c for c in ["candidate_id","source_table","classification","computation_primitive","readiness","review_status","evidence"] if c in candidates]].head(100))
    by_class = candidates.classification.value_counts().sort_index()
    ax = by_class.plot.bar(color="#356AA0", figsize=(7,3), legend=False)
    ax.set_title(f"Review-only candidates by class — n={len(candidates)} — exact")
    ax.set_ylabel("Candidates"); plt.xticks(rotation=25); plt.show()
assert not any(c.get("source_table") == "silver_model_predictions" for c in candidate_rows)


## Cross-source work orders

Cross-source repairs and semantic decisions are prioritized from each dossier's findings.

In [ ]:

work = READINESS.get("work_orders", [])
work_frame = pd.DataFrame(work)
if not work_frame.empty and "exactness" not in work_frame:
    work_frame["exactness"] = work_frame.table_name.map(ROWS.set_index("table_name").analysis_exactness) if "table_name" in work_frame else "exact"
display(work_frame.head(200) if not work_frame.empty else pd.DataFrame([{"status":"No work order emitted", "exactness": "exact"}]))


## Reproducibility and safety assertions

Campaign, registry, code, config, image, and artifact hashes plus hard scope assertions make the readout auditable.

In [ ]:

scope = READINESS["analysis_scope"]
assert scope["legacy_gold_read"] is False and scope["model_ready_read"] is False
assert scope["target_aware_analysis"] is False and scope["gold_contracts_included"] == 0
assert sorted(SPECS) == sorted(READINESS["expected_tables"])
model_row = ROWS.loc[ROWS.table_name.eq("silver_model_predictions")].iloc[0]
assert model_row.disposition == "excluded_leakage" and int(model_row.candidate_count) == 0
display(pd.DataFrame([{"assertion":"42/42 exact registry set", "passed":True, "exactness":"exact"},
                      {"assertion":"no Gold/model-ready/target analysis", "passed":True, "exactness":"exact"},
                      {"assertion":"model predictions quarantined with zero candidates", "passed":True, "exactness":"exact"},
                      {"assertion":"production feature configuration unchanged", "passed":True, "exactness":"exact"}]))
